# Phase 3: LoRA fine-tune + evaluation (Colab T4)
Runs: baseline eval of the untrained model, LoRA fine-tune, eval of the fine-tuned model, results table, and a results zip.

**How data and results move (DECISIONS D23):**
- **Data** comes from the git clone (`data/phase2/*.jsonl` are committed), checked against `data/manifest.json`. Nothing to upload to Drive.
- **Runs** (checkpoints, adapter, eval, logs, figures) are written straight to Drive: `My Drive/spice-verified-analog-finetuning/runs/phase3/<RUN_NAME>/`.
- **Results to your Mac:** the last cell writes a small `<RUN_NAME>_results.zip` (no weights) to Drive. Download it from drive.google.com, then run `python scripts/import_results_zip.py <zip>` on the Mac.

**Before running:** (1) `git push` your latest code and data; (2) in VS Code choose **Select Kernel -> Colab -> New Colab Server -> GPU -> T4**.
If Colab disconnects: reconnect, run all cells again. Training resumes from the last checkpoint and finished evaluations are skipped.

In [ ]:
# ===== CONFIG: change settings here only =====
CONFIG = dict(
    base_model="Qwen/Qwen2.5-0.5B-Instruct",   # main run: "Qwen/Qwen2.5-1.5B-Instruct" (also set per_device_batch=8, grad_accum=2)
    run_name="qwen25_05b_lora",
    epochs=2,
    learning_rate=2e-4,
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    per_device_batch=16, grad_accum=1,          # effective batch = product = 16
    precision="auto",                           # "auto" = fp16 (T4 has no bf16), falls back to fp32 if the loss is not finite
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO = "/content/spice-verified-analog-finetuning"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Keerthanareddy17/spice-verified-analog-finetuning.git", REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# torch comes preinstalled on Colab: do not reinstall it. ngspice is not needed in this notebook (scoring is a lookup).
# Colab ships an old torchao (0.10.0) that we never use; peft 0.21.2 aborts LoRA creation when it finds torchao < 0.16.
# Removing it is the safest fix (upgrading it could clash with Colab's torch).
!pip -q uninstall -y torchao
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps

In [ ]:
# Fail early, before any model loads: print versions and run peft's optional-package checks (the torchao check is what broke before).
import importlib, importlib.metadata as md, importlib.util
importlib.invalidate_caches()
for pkg in ["torch", "transformers", "trl", "peft", "accelerate", "datasets", "torchao", "bitsandbytes"]:
    try:
        print(f"{pkg:14s}", md.version(pkg))
    except md.PackageNotFoundError:
        print(f"{pkg:14s}", "not installed")
import peft.import_utils as piu
from peft import get_peft_model, LoraConfig   # noqa: F401
bad = []
for name in sorted(n for n in dir(piu) if n.startswith("is_") and n.endswith("_available")):
    try:
        getattr(piu, name)()
    except ImportError as e:
        bad.append(f"{name}: {e}")
    except TypeError:
        pass   # needs arguments, not an environment check
assert not bad, "peft found an incompatible optional package:\n" + "\n".join(bad)
print("peft optional-package checks OK")

In [ ]:
import sys
sys.path.insert(0, REPO + "/src")
import importlib, transformers, trl, peft, torch
from analogft import paths
from analogft.drive_check import verify_manifest
print("colab:", paths.IS_COLAB, "| data:", paths.DATA_DIR, "| runs:", paths.RUNS_DIR)
print("transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
ok, problems = verify_manifest(paths.DATA_DIR)
assert ok, "data/ in the repo clone does not match data/manifest.json (did you push the data commit? run git pull): " + "; ".join(problems[:5])
print("data manifest OK (data read from the git clone:", paths.DATA_DIR, ")")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("NO GPU: in VS Code use Select Kernel -> Colab -> New Colab Server -> GPU -> T4, then run all cells again.")
print("GPU:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
from analogft.finetune import FTConfig, train, evaluate_model
cfg = FTConfig(**CONFIG)
print("run directory:", cfg.run_dir)

## 1. Baseline: the untrained model (same prompts, same scoring)

In [ ]:
base_res = evaluate_model(cfg, "base", None, skip_if_done=True)

## 2. Fine-tune (watch: loss should fall quickly; the time estimate prints after 20 steps)

In [ ]:
summary = train(cfg)

## 3. Evaluate the fine-tuned model

In [ ]:
ft_res = evaluate_model(cfg, "finetuned", cfg.run_dir / "adapter", skip_if_done=True)

## 4. Results table (95% bootstrap CIs). Groq rows come from the committed results/phase3 files.

In [ ]:
from analogft.report import comparison_tables
t = comparison_tables(cfg.run_dir, cfg.run_name)
print("EVAL SUBSET (150 test examples, the main comparison)\n" + t["subset"])
print("\nFULL TEST SPLIT (375)\n" + t["test"])

In [ ]:
from IPython.display import Image, display
display(Image(str(cfg.run_dir / "figures" / "loss_curve.png")))

## 5. Save a small results zip to Drive (no checkpoints, no adapter) and list the run folder

In [ ]:
from analogft.resultszip import make_results_zip
z = make_results_zip(cfg.run_dir)
rel = z.relative_to(paths.STORAGE_ROOT)
print(f"wrote {z}  ({z.stat().st_size / 1e6:.2f} MB)")
print(f"On drive.google.com open:  My Drive / {paths.PROJECT_NAME} / {rel.as_posix()}")
print(f"then download it and on the Mac run:  python scripts/import_results_zip.py ~/Downloads/{z.name}")
print()
for p in sorted(cfg.run_dir.rglob("*")):
    if p.is_file() and "checkpoints" not in p.parts:
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(cfg.run_dir)}")